In [10]:
from sklearn.metrics import pairwise_distances, classification_report
from sklearn.metrics import silhouette_score
import pandas as pd
import numpy as np

In [11]:
df = pd.read_csv("/Users/milenaangelova/git-repo/EdgeCluster/results/synthetic/2-dim/tabular/stream 0/3/final_clustering.csv")

In [12]:
df.head()

,0,1,cluster,target,segment,stream
0,0.929782,0.651473,9,2,0,0
1,0.946655,0.653076,9,2,0,0
2,0.352081,0.538320,9,1,0,0
3,0.354808,0.534828,9,1,0,0
4,0.368887,0.513268,9,1,0,0


In [13]:
data = df[df.columns[:-4]]
target = df["target"]
cluster = df["cluster"]

In [14]:
def calculate_distances(data):
    return pairwise_distances(data,data)

def find_nearest_neighbors(sample, neighbors, n_neighbors):
    """Finds the nearest neighbors to the given sample"""
    neighbors = np.delete(neighbors, sample)
    nearest_neibour_indexes = np.argsort(neighbors)[:n_neighbors]
    return nearest_neibour_indexes

def get_connectivity(dataframe, i, j, j_index):
    """Gets the connectivity value from a given dataframe"""
    i_class = dataframe.iloc[i]["cluster"]
    j_class = dataframe.iloc[j]["cluster"]
    state = (0 if i_class == j_class else float(1)/(j_index + 1))
    return state

def connectivity_samples(X, y, n_neighbors, metric=None, distance_matrix=None, **kwds):
    """Calculates the connectivity for each sample in the dataset."""
    dataframe = pd.DataFrame(data=X, index=range(len(X)))
    dataframe["cluster"] = y
    N = len(X)
    
    if distance_matrix is None:
        distance_matrix = pairwise_distances(X, metric=metric, **kwds)
    
    connectivity_list = []
    for i in range(N):
        connectivity_sum = 0
        nearest_neighbors = find_nearest_neighbors(i, distance_matrix[i], n_neighbors)
        for j in range(n_neighbors):
            connectivity_sum += get_connectivity(dataframe, i, nearest_neighbors[j], j)
        connectivity_list.append(connectivity_sum)
    return connectivity_list

def calculate_connectivity(X_train, y_train, columns, n_neighbors, metric=None, distance_matrix=None):
    si_samples = connectivity_samples(X_train, y_train, n_neighbors, metric, distance_matrix)
    dataframe = pd.DataFrame(data=X_train, columns=columns, index=range(0, len(y_train)))
    dataframe["CONN"] = si_samples
    dataframe["cluster"] = y_train
    #sorted_dataframe = dataframe.sort_values(["CONN"], ascending=[True])
    return dataframe

# F1
def f_measure(pred, true):
    value = (2 * len(pred & true)) / (len(true) + len(pred))
    return value

# SI
def calculate_silhouette(data, clusters):
    distances = calculate_distances(data.to_numpy(copy=True))
    return silhouette_score(distances, clusters, metric='precomputed')

In [15]:
def evalutation_report(data, pred_labels, true_labels):
    distances = calculate_distances(data.to_numpy(copy=True))
    connectivity = calculate_connectivity(data, 
                                        pred_labels,
                                        [x for x in range(data.shape[1])],
                                        10, distance_matrix=distances)['CONN'].sum()
    
    F1 = f_measure(pred_labels, true_labels)
    if len(pred_labels.unique()) > 1:
        SI = calculate_silhouette(data, pred_labels)
    else:
        SI = None

    return connectivity, F1, SI

In [16]:
connectivity, F1, SI = evalutation_report(data=data, pred_labels=cluster, true_labels=target)

In [17]:
metrics_df =  pd.DataFrame({
    'connectivity': [connectivity],
    'F1': [F1],
    'SI': [SI]
})
metrics_df.head()

,connectivity,F1,SI
0,0,1.0,None
